# 25 - Poincaré Duality Verification & Manifold Recognition

A topological space $X$ is called a **closed orientable $n$-manifold** if and only if it satisfies Poincaré duality: for every degree $k$, the cap product with the fundamental class $[M] \in H_n(M;\mathbb{Z})$ gives an isomorphism
$$\cap [M] : H^k(M;\mathbb{Z}) \xrightarrow{\cong} H_{n-k}(M;\mathbb{Z}).$$

This is not just a theorem — it is a *certificate*. If you have a simplicial or CW complex and want to know whether it is a manifold (rather than a pseudomanifold, a complex with wrong-dimensional links, or a space with boundary), checking Poincaré duality is the right test.

## Learning Goals
- Understand why Poincaré duality characterises closed orientable manifolds
- Use `detect_poincare_dimension` to certify manifold status
- Compute the cap product explicitly with `simplicial_cap_product`
- Construct the full duality isomorphism with `compute_poincare_duality_map`
- Extract the intersection pairing chain for use in NB 07 (intersection forms)
- Understand failure cases: spaces with boundary, non-orientable spaces, pseudomanifolds

## Three-Level View
| Level | Object | Tool |
|---|---|---|
| **Geometric** | Triangulated closed orientable $n$-manifold $M$ | `SimplicialComplex` |
| **Algebraic** | Cap product pairing $\cap [M]: H^k \to H_{n-k}$ | `compute_poincare_duality_map` |
| **Result** | `PoincareDualityCertificate` with `is_manifold`, `dimension`, `exact` | `detect_poincare_dimension` |

## Formal Background

**Cap product**: For $\alpha \in C^k(X)$ a cochain and $\sigma \in C_n(X)$ a chain, the cap product $\alpha \cap \sigma \in C_{n-k}(X)$ is defined by
$$\alpha \cap \sigma = \alpha(\sigma_{[v_0,\ldots,v_k]}) \cdot \sigma_{[v_k,\ldots,v_n]}.$$
Poincaré duality says this descends to an isomorphism on homology when $\sigma = [M]$.

In [ ]:
import numpy as np
from pysurgery.core.poincare_duality_verification import (
    PoincareDualityCertificate,
    detect_poincare_dimension,
    is_poincare_duality_complex,
    simplicial_cap_product,
    compute_poincare_duality_map,
    extract_intersection_pairing_chain,
)
from pysurgery.core.complexes import SimplicialComplex
import matplotlib.pyplot as plt

print("=" * 70)
print("25 - Poincaré Duality Verification & Manifold Recognition: Ready")
print("=" * 70)

## Part 1: What Poincaré Duality Says

For the torus $T^2$, we have:
$$H^0(T^2) \cong H_2(T^2) \cong \mathbb{Z}, \quad H^1(T^2) \cong H_1(T^2) \cong \mathbb{Z}^2, \quad H^2(T^2) \cong H_0(T^2) \cong \mathbb{Z}.$$
The cap product with $[T^2]$ provides the isomorphism in each degree. Notice the symmetry: the Betti numbers of a Poincaré duality space are palindromic, $\beta_k = \beta_{n-k}$.

For $\mathbb{R}P^2$ (non-orientable): Poincaré duality holds with $\mathbb{Z}/2$ coefficients but *not* with $\mathbb{Z}$ coefficients — the fundamental class does not exist integrally. `detect_poincare_dimension` checks the *integral* duality by default.

In [ ]:
# Standard 2-manifolds as test cases
spaces = {
    "T²  (torus)":          SimplicialComplex.torus(),
    "S²  (sphere)":         SimplicialComplex.sphere(2),
    "S³  (3-sphere)":       SimplicialComplex.sphere(3),
    "RP² (non-orientable)": SimplicialComplex.real_projective_plane(),
}

print(f"{'Space':<22} {'is_manifold':<13} {'dim':<6} {'orientable':<12} {'exact'}")
print("-" * 70)
for name, X in spaces.items():
    cert = detect_poincare_dimension(X)
    print(f"{name:<22} {str(cert.is_manifold):<13} {str(cert.dimension):<6} "
          f"{str(cert.orientable):<12} {cert.exact}")

## Part 2: The `PoincareDualityCertificate` Contract

`detect_poincare_dimension` returns a rich certificate:

| Field | Meaning |
|---|---|
| `is_manifold` | `True` iff Poincaré duality holds integrally in all degrees |
| `dimension` | The detected manifold dimension (or `None` if not a manifold) |
| `orientable` | `True` iff $H_n(M;\mathbb{Z}) \cong \mathbb{Z}$ (fundamental class exists) |
| `betti_numbers` | List of Betti numbers (palindromic iff manifold) |
| `duality_isomorphisms` | Per-degree matrices of the cap product maps |
| `exact` | Certified by `compute_modular_rank_certificate` internally |
| `theorem_tag` | Algorithm identifier |

The `is_poincare_duality_complex(cw, dimension)` function is a lighter boolean check when you already know the expected dimension.

In [ ]:
T2 = SimplicialComplex.torus()
cert: PoincareDualityCertificate = detect_poincare_dimension(T2)

print("PoincareDualityCertificate for T²:")
print(f"  is_manifold:           {cert.is_manifold}")
print(f"  dimension:             {cert.dimension}")
print(f"  orientable:            {cert.orientable}")
print(f"  betti_numbers:         {cert.betti_numbers}")
print(f"  exact:                 {cert.exact}")
print(f"  theorem_tag:           {cert.theorem_tag}")
print()

# Palindrome check
betti = cert.betti_numbers
n = cert.dimension
palindrome = all(betti[k] == betti[n - k] for k in range(n + 1))
print(f"  Betti palindrome check (β_k = β_{{n-k}}): {palindrome}")

# Quick boolean check (lighter, no certificate returned)
print()
quick = is_poincare_duality_complex(T2, dimension=2)
print(f"  is_poincare_duality_complex(T², dim=2): {quick}")

## Part 3: The Cap Product Explicitly

The cap product $\alpha \cap [M]$ is the algebraic mechanism behind Poincaré duality. For a concrete triangulation, `simplicial_cap_product` computes it as a chain map.

**Topological meaning**: if $\alpha$ is a cohomology class Poincaré-dual to a submanifold $N \subset M$, then $\alpha \cap [M] = [N]$ (the homology class of $N$).

In [ ]:
# Pick a degree-1 cocycle on T²
# On T², H¹ = Z², generated by the two longitude/meridian classes

fundamental_class = T2.fundamental_cycle()  # [T²] ∈ C₂(T²)
cocycle_1 = T2.representative_cocycle(degree=1)  # a 1-cocycle

print(f"Fundamental class [T²]: {len(fundamental_class)} nonzero simplices")
print(f"Representative cocycle: type={type(cocycle_1).__name__}")

# Compute the cap product: α ∩ [T²] ∈ C₁(T²)
cap_result = simplicial_cap_product(T2, cocycle_1, fundamental_class, dimension=1)
print(f"Cap product result (1-chain): {cap_result}")
print()

# The cap product of [T²] with itself (degree-2 class) gives a 0-cycle
cocycle_2 = T2.representative_cocycle(degree=2)
cap_top = simplicial_cap_product(T2, cocycle_2, fundamental_class, dimension=2)
print(f"Top-degree cap product (0-chain): {cap_top}")
print("This should be a single vertex (the fundamental class of a point)")

## Part 4: The Full Duality Map

`compute_poincare_duality_map` returns the full matrix of the isomorphism $\cap [M]: H^k(M) \to H_{n-k}(M)$ for all $k$. Its shape is (rank of $H^k$) × (rank of $H_{n-k}$) and it should be invertible (square and full rank) for a manifold.

In [ ]:
# Compute the full Poincaré duality isomorphism for T²
duality_maps = compute_poincare_duality_map(T2, dimension=2)

for k, D_k in enumerate(duality_maps):
    if D_k is not None:
        rank = np.linalg.matrix_rank(D_k.astype(float))
        shape = D_k.shape
        is_iso = (shape[0] == shape[1] == rank)
        print(f"  D_{k}: shape={shape}, rank={rank}, isomorphism={is_iso}")

# Visualise the duality maps
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for k, (D_k, ax) in enumerate(zip(duality_maps, axes)):
    if D_k is None or D_k.size == 0:
        ax.text(0.5, 0.5, f"D_{k} = trivial", ha="center", va="center",
                transform=ax.transAxes)
        ax.set_title(f"D_{k}: H^{k} → H_{{{2-k}}}")
        continue
    im = ax.imshow(D_k, cmap="RdBu", aspect="auto",
                   vmin=-np.max(np.abs(D_k)), vmax=np.max(np.abs(D_k)))
    ax.set_title(f"D_{k}: H^{k} → H_{{{2-k}}}")
    ax.set_xlabel("H₁ basis"); ax.set_ylabel(f"H^{k} basis")
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.suptitle("Poincaré Duality Maps for T²", y=1.02)
plt.tight_layout(); plt.show()

## Part 5: Intersection Pairing Chain

The **intersection pairing** on a $2n$-manifold is a symmetric (or skew-symmetric) bilinear form on $H_n(M;\mathbb{Z})$:
$$\lambda: H_n(M;\mathbb{Z}) \times H_n(M;\mathbb{Z}) \to \mathbb{Z}, \quad \lambda(\alpha, \beta) = \langle PD(\alpha), \beta \rangle$$
where $PD: H_n \to H^n$ is the Poincaré duality isomorphism. This is the intersection form of NB 07 — but now derived *from the geometry* rather than specified by hand.

`extract_intersection_pairing_chain` returns this pairing as a matrix, ready to be passed to `IntersectionForm`.

In [ ]:
from pysurgery.core.intersection_forms import IntersectionForm

# Extract intersection pairing from a 4-manifold: CP²
CP2 = SimplicialComplex.complex_projective_space(2)
cert_CP2 = detect_poincare_dimension(CP2)
print(f"CP²: is_manifold={cert_CP2.is_manifold}, dim={cert_CP2.dimension}")
print(f"     Betti numbers: {cert_CP2.betti_numbers}")

pairing_matrix = extract_intersection_pairing_chain(CP2, dimension=4)
print(f"Intersection pairing matrix:\n{pairing_matrix}")

# Feed directly into IntersectionForm for signature computation
iform = IntersectionForm(matrix=pairing_matrix)
print(f"Signature σ(CP²) = {iform.signature()}")   # should be +1
print(f"is_definite = {iform.is_definite}")
print(f"is_unimodular = {iform.is_unimodular}")

## Part 6: Failure Cases — When Duality Fails

Poincaré duality fails for:
1. **Spaces with boundary** (e.g. disk $D^n$): boundary breaks the duality pairing
2. **Non-orientable manifolds** (e.g. $\mathbb{R}P^2$): no integral fundamental class
3. **Pseudomanifolds** with non-manifold points (e.g. pinched torus): links are wrong
4. **Infinite spaces**: compactness is required for the fundamental class

In [ ]:
# Case 1: Disk D² — has boundary, not a manifold
D2 = SimplicialComplex.disk(2)
cert_D2 = detect_poincare_dimension(D2)
print(f"D²: is_manifold={cert_D2.is_manifold}")
if cert_D2.obstruction:
    print(f"    obstruction: {cert_D2.obstruction}")

# Case 2: Pinched torus (identify two vertices of T²) — pseudomanifold
try:
    pinched = SimplicialComplex.pinched_torus()
    cert_pinched = detect_poincare_dimension(pinched)
    print(f"Pinched T²: is_manifold={cert_pinched.is_manifold}")
    if cert_pinched.obstruction:
        print(f"    obstruction: {cert_pinched.obstruction}")
except AttributeError:
    print("Pinched torus constructor not yet available; skipping")

# Case 3: Cone over S² — contractible, Betti numbers wrong
cone_S2 = SimplicialComplex.cone(SimplicialComplex.sphere(2))
cert_cone = detect_poincare_dimension(cone_S2)
print(f"Cone(S²): is_manifold={cert_cone.is_manifold}, Betti={cert_cone.betti_numbers}")

print()
print("Summary: detect_poincare_dimension gives a CERTIFIED manifold test.")
print("If exact=True and is_manifold=True, the space is provably a homology manifold.")

## Summary Checklist

- [x] Understood Poincaré duality as the algebraic characterisation of closed orientable manifolds
- [x] Used `detect_poincare_dimension` to certify manifold status with `PoincareDualityCertificate`
- [x] Verified `is_manifold`, `dimension`, `orientable`, and `exact` fields
- [x] Computed `simplicial_cap_product` explicitly on the torus
- [x] Constructed the full duality isomorphism matrices with `compute_poincare_duality_map`
- [x] Extracted intersection pairing chains for input to `IntersectionForm`
- [x] Understood failure modes for non-manifolds and pseudomanifolds

## Next Steps
- **NB 07**: Use the intersection pairing extracted here as the starting point for intersection form analysis
- **NB 28**: Handle decompositions also produce intersection forms — compare the two derivations
- **NB 34**: The capstone uses `detect_poincare_dimension` as step 2 of the surgery pipeline